In [1]:

import logging
import time
import sys
from typing import List, Optional

from selenium import webdriver
from selenium.webdriver.common.by import By
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC
from selenium.webdriver.common.keys import Keys
from selenium.common.exceptions import TimeoutException, NoSuchElementException, ElementClickInterceptedException, StaleElementReferenceException
from selenium.webdriver.chrome.options import Options
from selenium.webdriver.chrome.service import Service
from webdriver_manager.chrome import ChromeDriverManager


logging.basicConfig(
    level=logging.INFO,
    format='%(asctime)s - %(levelname)s - %(message)s',
    handlers=[
        logging.StreamHandler(sys.stdout),
        logging.FileHandler('soilhive_automation.log')
    ]
)
logger = logging.getLogger(__name__)

In [2]:
NAME_ALIASES = {
    "Cabo Verde":               "Cape Verde",
    "Czech Republic":           "Czechia",
    "South Korea":              "Korea",
    "North Korea":              "Korea",
    "Ivory Coast":              "Cote d'Ivoire",
    "Timor-Leste":              "East Timor",
    "Eswatini":                 "Swaziland",
    "Republic of the Congo":    "Congo",
    "Democratic Republic of the Congo": "Congo",
    "United Arab Emirates":     "Emirates",
    "Saint Kitts and Nevis":    "Saint Kitts",
    "Saint Vincent and the Grenadines": "Saint Vincent",
    "Sao Tome and Principe":    "Sao Tome e Principe",
    "Turkey":                   "Türkiye",
    "Palestine":                "Palestine",
    "Kosovo":                   "Kosova",
}


class SoilHiveAutomator:
    def __init__(self, headless: bool = False, download_dir: str = None):
        self.headless = headless
        self.download_dir = download_dir
        self.driver = self.setup_driver()
        self.base_url = "https://soilhive.ag/app/availability"

    def _wait(self, timeout: int = 15):
        return WebDriverWait(self.driver, timeout)

    def setup_driver(self) -> webdriver.Chrome:
        logger.info("Setting up Chrome driver...")
        chrome_options = Options()
        if self.headless:
            chrome_options.add_argument("--headless=new")
        chrome_options.add_argument("--no-sandbox")
        chrome_options.add_argument("--disable-dev-shm-usage")
        chrome_options.add_argument("--window-size=1920,1080")
        chrome_options.add_argument("--start-maximized")
        chrome_options.add_argument("--disable-blink-features=AutomationControlled")
        chrome_options.add_experimental_option("excludeSwitches", ["enable-automation"])
        chrome_options.add_experimental_option("useAutomationExtension", False)
        if self.download_dir:
            prefs = {"download.default_directory": self.download_dir}
            chrome_options.add_experimental_option("prefs", prefs)
        try:
            service = Service(ChromeDriverManager().install())
            driver = webdriver.Chrome(service=service, options=chrome_options)
            driver.set_page_load_timeout(45)
            return driver
        except Exception as e:
            logger.error(f"Failed to setup driver: {e}")
            raise

    def close(self):
        if self.driver:
            logger.info("Closing driver...")
            self.driver.quit()

    def take_screenshot(self, name: str):
        filename = f"screenshot_{name}_{int(time.time())}.png"
        self.driver.save_screenshot(filename)
        logger.info(f"Screenshot saved: {filename}")

    def load_page(self):
        logger.info(f"Navigating to {self.base_url}")
        try:
            self.driver.get(self.base_url)
        except Exception:
            pass
        try:
            self._wait(15).until(EC.element_to_be_clickable(
                (By.CSS_SELECTOR, ".coi-banner__accept"))).click()
            logger.info("Cookies accepted.")
        except TimeoutException:
            pass
        try:
            self._wait(15).until(EC.element_to_be_clickable(
                (By.XPATH, "//button[contains(text(), 'Start')]"))).click()
            logger.info("Clicked 'Start'.")
        except TimeoutException:
            pass

    def _restart_driver(self):
        logger.warning("Restarting driver...")
        try:
            self.driver.quit()
        except Exception:
            pass
        self.driver = self.setup_driver()
        self.load_page()

    def _click_suggestion(self, keywords: list, max_retries: int = 6) -> Optional[str]:
        sug_loc = (By.CSS_SELECTOR, ".va-location__suggestions-item")
        for _ in range(max_retries):
            try:
                elements = self.driver.find_elements(*sug_loc)
                if not elements:
                    time.sleep(0.4)
                    continue
                for el in elements:
                    try:
                        text = el.text
                        if any(kw in text.lower() for kw in keywords):
                            self.driver.execute_script("arguments[0].click();", el)
                            return text.strip()
                    except StaleElementReferenceException:
                        break
            except StaleElementReferenceException:
                time.sleep(0.3)
        return None

    def _wait_page_loaded(self, timeout: int = 90):
        """Wait for 'Loading results' to disappear.
        If loading never finishes within timeout, log a warning and proceed anyway.
        """
        try:
            WebDriverWait(self.driver, 8).until(
                lambda d: "Loading results" in d.page_source)
            logger.info("  Loading detected — waiting for completion...")
        except TimeoutException:
            logger.info("  No loading indicator — assuming ready.")
            time.sleep(2)
            return

        try:
            WebDriverWait(self.driver, timeout).until(
                lambda d: "Loading results" not in d.page_source)
            logger.info("  Page fully loaded.")
        except TimeoutException:
            # Loading never finished — still try to proceed (site may still be usable)
            logger.warning(f"  Loading still active after {timeout}s — proceeding anyway.")
        time.sleep(1)

    def search_country(self, country_name: str):
        aliases = [country_name]
        if country_name in NAME_ALIASES:
            aliases.append(NAME_ALIASES[country_name])

        for attempt_name in aliases:
            try:
                logger.info(f"  Typing: '{attempt_name}'")
                inp = self._wait(15).until(
                    EC.element_to_be_clickable((By.CSS_SELECTOR, "input#search")))
                inp.clear()
                time.sleep(0.5)
                inp.send_keys(attempt_name)

                sug_loc = (By.CSS_SELECTOR, ".va-location__suggestions-item")
                try:
                    self._wait(10).until(EC.presence_of_element_located(sug_loc))
                except TimeoutException:
                    logger.warning(f"  No suggestions for '{attempt_name}'")
                    inp.clear()
                    continue

                keywords = [attempt_name.lower()] + [p.lower() for p in attempt_name.split()]
                clicked = self._click_suggestion(keywords)
                if not clicked:
                    logger.warning(f"  No match for '{attempt_name}'")
                    inp.clear()
                    continue

                logger.info(f"  Clicked: '{clicked}'")
                self._wait_page_loaded(timeout=90)
                return

            except Exception as e:
                logger.error(f"  Error with '{attempt_name}': {e}")
                continue

        raise NoSuchElementException(f"Could not find '{country_name}' on SoilHive")

    def select_all_datasets(self):
        try:
            logger.info("Selecting all datasets...")
            el = self._wait(20).until(EC.element_to_be_clickable(
                (By.CSS_SELECTOR, ".va-ds__select-all .va-checkbox")))
            self.driver.execute_script("arguments[0].scrollIntoView({block: 'center'});", el)
            self.driver.execute_script("arguments[0].click();", el)
            logger.info("Clicked 'Select all'.")
        except Exception as e:
            logger.error(f"Error selecting datasets: {e}")
            self.take_screenshot("select_datasets_error")
            raise

    def click_download(self):
        try:
            logger.info("Clicking Download...")
            btn = self._wait(20).until(EC.element_to_be_clickable(
                (By.XPATH, "//button[contains(text(), 'Download')]")))
            self.driver.execute_script("arguments[0].scrollIntoView({block: 'center'});", btn)
            self.driver.execute_script("arguments[0].click();", btn)
            logger.info("Download clicked.")
        except Exception as e:
            logger.error(f"Error clicking download: {e}")
            self.take_screenshot("click_download_error")
            raise

    def fill_email_and_submit(self, email: str):
        try:
            logger.info(f"Filling email: {email}")
            inp = self._wait(20).until(
                EC.visibility_of_element_located((By.CSS_SELECTOR, "input#email")))
            inp.clear()
            inp.send_keys(email)
            chk = self.driver.find_element(
                By.CSS_SELECTOR, ".va-ds-download__content label.va-checkbox")
            self.driver.execute_script("arguments[0].scrollIntoView({block: 'center'});", chk)
            self.driver.execute_script("arguments[0].click();", chk)
            logger.info("Terms accepted.")
            submit = self._wait(20).until(EC.element_to_be_clickable(
                (By.CSS_SELECTOR, "button.download-data-request-submit-button")))
            self.driver.execute_script("arguments[0].click();", submit)
            time.sleep(3)
            logger.info("Request submitted.")
        except Exception as e:
            logger.error(f"Error in email/submit: {e}")
            self.take_screenshot("form_error")
            raise

    def process_country(self, country: str, email: str):
        try:
            self.search_country(country)
            self.select_all_datasets()
            self.click_download()
            self.fill_email_and_submit(email)
            try:
                self.driver.refresh()
                self._wait(15).until(
                    EC.presence_of_element_located((By.CSS_SELECTOR, "input#search")))
                time.sleep(1)
            except Exception as ref_err:
                logger.warning(f"Refresh failed — restarting driver")
                self._restart_driver()
        except Exception as e:
            err = str(e)
            logger.error(f"FAILED '{country}': {err[:120]}")
            if any(k in err for k in ["invalid session", "no such window", "disconnected",
                                       "timeout", "Timeout", "HTTPConnection"]):
                self._restart_driver()
            else:
                try:
                    self.driver.get(self.base_url)
                    self._wait(15).until(
                        EC.presence_of_element_located((By.CSS_SELECTOR, "input#search")))
                except Exception:
                    self._restart_driver()


In [3]:
import os

def main():
    # Countries that still need requests (all others already submitted or skipped)
    remaining = [
        "Kosovo", "Palestine", "Turkey",
        "Sao Tome and Principe",
        # Re-run any that may have had cascade failures
        "Afghanistan", "Algeria", "Angola", "Antigua and Barbuda",
        "Bahamas", "Barbados", "Bangladesh", "Belize", "Benin", "Bhutan",
        "Botswana", "Brunei", "Burkina Faso", "Burundi",
        "Cambodia", "Central African Republic", "Chad", "China",
        "Comoros", "Cuba", "Dominican Republic", "El Salvador",
        "Estonia", "Eswatini", "Ethiopia",
        "France", "Gabon", "Germany", "Ghana", "Greece", "Guatemala",
        "Guinea", "Guinea-Bissau", "Haiti",
        "Iran", "Iraq", "Israel", "Ivory Coast",
        "Jamaica", "Jordan", "Kazakhstan", "Kenya", "Kuwait", "Kyrgyzstan",
        "Laos", "Lesotho", "Liberia",
        "Madagascar", "Malawi", "Maldives", "Mali", "Mauritania", "Mauritius",
        "Moldova", "Monaco", "Montenegro", "Myanmar",
        "Netherlands", "Nicaragua", "Nigeria",
        "Oman", "Panama", "Paraguay",
        "Russia", "Rwanda",
        "Saint Kitts and Nevis", "San Marino", "Seychelles", "Sierra Leone",
        "South Sudan", "Sri Lanka", "Sudan", "Suriname",
        "Taiwan", "Tajikistan", "Tanzania", "Togo",
        "Trinidad and Tobago", "Tunisia", "Turkmenistan",
        "Uganda", "Uzbekistan", "Vatican City", "Venezuela",
        "Zambia", "Zimbabwe",
        "Croatia", "Estonia", "France", "Germany", "Greece", "Kosovo",
        "Liechtenstein", "Malta", "Moldova", "Monaco", "Montenegro",
        "Netherlands", "Russia", "San Marino", "Vatican City",
    ]
    # Deduplicate
    remaining = list(dict.fromkeys(remaining))

    # Skip countries that already have data folders
    DATA_DIR = "/home/agbelgaid/Documents/WORKSPACE/DataCollection/SoilHive/data"
    NAME_MAP = {
        "czech republic":  "czechia",
        "south korea":     "korea, republic of",
        "vietnam":         "viet nam",
        "bolivia":         "bolivia, plurinational state of",
        "ivory coast":     "cote d'ivoire",
    }
    existing_folders = {
        d.replace("_", " ").lower()
        for d in os.listdir(DATA_DIR)
        if os.path.isdir(os.path.join(DATA_DIR, d))
        and not d.startswith(("Z", "Y"))
    }
    countries = []
    skipped = []
    for c in remaining:
        norm = c.lower()
        mapped = NAME_MAP.get(norm, norm)
        if mapped in existing_folders or norm in existing_folders:
            skipped.append(c)
        else:
            countries.append(c)

    logger.info(f"Remaining to scrape : {len(countries)}")
    logger.info(f"Already have data   : {len(skipped)} — skipped")

    email = "dsconite@gmail.com"
    automator = SoilHiveAutomator(headless=True)

    try:
        automator.load_page()
        for i, country in enumerate(countries, 1):
            logger.info(f"[{i}/{len(countries)}] Processing: {country}")
            automator.process_country(country, email)
    except Exception as e:
        logger.critical(f"Critical error: {e}")
    finally:
        automator.close()


In [4]:
main()

2026-03-17 12:26:40,890 - INFO - Remaining to scrape : 93


2026-03-17 12:26:40,892 - INFO - Already have data   : 0 — skipped


2026-03-17 12:26:40,892 - INFO - Setting up Chrome driver...


2026-03-17 12:26:40,893 - INFO - ====== WebDriver manager ======


2026-03-17 12:26:40,967 - INFO - Get LATEST chromedriver version for google-chrome


...

2026-03-17 13:01:06,407 - INFO - Download clicked.


2026-03-17 13:01:06,408 - INFO - Filling email: dsconite@gmail.com


2026-03-17 13:01:06,554 - INFO - Terms accepted.


2026-03-17 13:01:09,580 - INFO - Request submitted.


2026-03-17 13:01:12,990 - INFO - Closing driver...
